# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [87]:
# Write your code below.
%load_ext dotenv
%dotenv 


The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [88]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [89]:
import os
from glob import glob

# Write your code below.
parquet_files = glob(os.path.join( os.getenv('PRICE_DATA'), "**/*.parquet"), recursive = True)
for file_path in parquet_files:
        print(file_path)


        



../../05_src/data/prices/IMV/IMV_2019/part.0.parquet
../../05_src/data/prices/IMV/IMV_2019/part.1.parquet
../../05_src/data/prices/IMV/IMV_2017/part.0.parquet
../../05_src/data/prices/IMV/IMV_2017/part.1.parquet
../../05_src/data/prices/IMV/IMV_2010/part.0.parquet
../../05_src/data/prices/IMV/IMV_2010/part.1.parquet
../../05_src/data/prices/IMV/IMV_2011/part.0.parquet
../../05_src/data/prices/IMV/IMV_2011/part.1.parquet
../../05_src/data/prices/IMV/IMV_2016/part.0.parquet
../../05_src/data/prices/IMV/IMV_2016/part.1.parquet
../../05_src/data/prices/IMV/IMV_2020/part.0.parquet
../../05_src/data/prices/IMV/IMV_2020/part.1.parquet
../../05_src/data/prices/IMV/IMV_2018/part.0.parquet
../../05_src/data/prices/IMV/IMV_2018/part.1.parquet
../../05_src/data/prices/IMV/IMV_2009/part.0.parquet
../../05_src/data/prices/IMV/IMV_2009/part.1.parquet
../../05_src/data/prices/IMV/IMV_2013/part.0.parquet
../../05_src/data/prices/IMV/IMV_2013/part.1.parquet
../../05_src/data/prices/IMV/IMV_2014/part.0.p

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [90]:
# Write your code below.      

#Adding lags
dd_px = dd.read_parquet(parquet_files).set_index("ticker")
dd_feat = dd_px.groupby('ticker', group_keys=False).apply(
    lambda x: x.sort_values('Date').assign( 
        Close_lag_1=x['Close'].shift(1),
        Adj_Close_lag_1=x['Adj Close'].shift(1)
    ),
    meta=dd_px._meta.assign(Close_lag_1='float64', Adj_Close_lag_1='float64')
)

#Adding returns
if 'Close_lag_1' in dd_feat.columns and 'Close' in dd_feat.columns: # Check if columns exist
    dd_feat['returns'] = (dd_feat['Close'] / dd_feat['Close_lag_1']) - 1

# Calculate hi_lo_range and add to dd_feat
if 'High' in dd_feat.columns and 'Low' in dd_feat.columns: # Check if columns exist
    dd_feat['hi_lo_range'] = dd_feat['High'] - dd_feat['Low']


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [91]:
# Write your code below.

df_feat = dd_feat.compute()

# Sort by Ticker and Date before applying rolling function
df_feat = df_feat.sort_values(by=['Ticker', 'Date'])

# Add 10-day rolling mean of 'returns' for each 'Ticker'
if 'returns' in df_feat.columns:
    df_feat['returns_rolling_10d_mean'] = (
        df_feat.groupby('Ticker')['returns']
        .rolling(window=10, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )

#Im really stumped on this one (sad face) any help would be appreciated

KeyError: 'Ticker'

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

#Using pandas is easier as Dask creates partitions which may not be ideal for rolling operations?

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.